<a href="https://colab.research.google.com/github/LuciaKajanova/dspracticum25_flowers_team/blob/CNN_HW2/Kvetinky_SimpleCNN_HW2_run.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Let's build a simple neural network to classify images from the FashionMNIST dataset.

**1. Import Libraries**

In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torchvision import datasets, transforms

*Checking for GPU Availability*

This code checks if a CUDA-enabled GPU is available and sets the `device` accordingly. If no GPU is available, it defaults to the CPU.

In [2]:
# Check if GPU is available
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')

Using device: cuda


**2. Data Preparation**

In [3]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [4]:
# Define a transform to convert images to tensors
transform = transforms.Compose([
    transforms.Resize((28, 28)),
    transforms.ToTensor(),
])

# Download and load the training data

train_set = datasets.ImageFolder(root="/content/drive/My Drive/datasetFlower/dataset/train", transform=transform)
train_loader = torch.utils.data.DataLoader(train_set, batch_size=64, shuffle=True)

# Download and load the test data

test_set = datasets.ImageFolder(root="/content/drive/My Drive/datasetFlower/dataset/test", transform=transform)
test_loader = torch.utils.data.DataLoader(test_set, batch_size=64, shuffle=False)

**3. Neural Network Model**

přidána jedna vrstva

In [7]:
class SimpleCNN(nn.Module):
    def __init__(self):
        super(SimpleCNN, self).__init__()
        self.conv1 = nn.Conv2d(in_channels=3, out_channels=32, kernel_size=3)
        self.conv2 = nn.Conv2d(in_channels=32, out_channels=64, kernel_size=3)
        self.flat = nn.Flatten()
        self.fc1 = nn.Linear(in_features=24*24*64, out_features=256)
        # nová mezivrstva a dropout presunut sem
        self.fc_1_5 = nn.Linear(in_features=256, out_features=128)
        self.drop = nn.Dropout(0.25)
        # výstupní vrstva
        self.fc2 = nn.Linear(in_features=128, out_features=10)

    def forward(self, x):
        x = F.relu(self.conv1(x))
        x = F.relu(self.conv2(x))
        x = self.flat(x)

        x = F.relu(self.fc1(x))
        x = F.relu(self.fc_1_5(x))  # nová vrstva
        x = self.drop(x)

        x = self.fc2(x)
        return x

model = SimpleCNN().to(device)
model


SimpleCNN(
  (conv1): Conv2d(3, 32, kernel_size=(3, 3), stride=(1, 1))
  (conv2): Conv2d(32, 64, kernel_size=(3, 3), stride=(1, 1))
  (flat): Flatten(start_dim=1, end_dim=-1)
  (fc1): Linear(in_features=36864, out_features=256, bias=True)
  (fc_1_5): Linear(in_features=256, out_features=128, bias=True)
  (drop): Dropout(p=0.25, inplace=False)
  (fc2): Linear(in_features=128, out_features=10, bias=True)
)

**4. Loss & Optimizer**

(-Adam metoda si pamatuje „směr“ posledních gradientů a dává jim váhu.

-Když gradienty směřují pořád stejným směrem, krok je větší – což urychluje konvergenci.

-Když gradienty mění směr, krok se zmenší – což stabilizuje učení a brání oscilacím.)

In [8]:
criterion = nn.CrossEntropyLoss()
# Adam optimizer: adaptivní učení s momenty, často rychlejší konvergence než SGD
optimizer = optim.Adam(model.parameters(), lr=0.002)

**5. Training loop**  
(přidán early stopping: trénink se zastaví, pokud se test loss po `patience` epochách nezlepšuje o více než `threshold`

Současně se loss počítá jako průměr přes všechny batche, takže výsledná hodnota není náhodně rozkolísaná, ale vyjadřuje celkový trend během celé epochy.)


In [9]:
patience = 3          # Počet epoch bez zlepšení, po kterých se trénink zastaví (můžeme si s tím hrát – větší hodnota = delší čekání)
threshold = 0.005     # Minimální zlepšení testovací loss, aby se považovala za lepší (můžeme si hrát – menší číslo = přísnější kontrola ale delší čekání)

best_loss = float('inf')
epochs_no_improve = 0

for epocha in range(50):  # Trénujeme max 50 epoch, early stopping to může ukončit dříve
    running_loss = 0.0
    for images, labels in train_loader:
        images = images.to(device)
        labels = labels.to(device)

        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item()

    train_loss = running_loss / len(train_loader)

    # Vyhodnocení na testovacím datasetu
    test_loss = 0.0
    with torch.no_grad():
        for images, labels in test_loader:
            images = images.to(device)
            labels = labels.to(device)
            outputs = model(images)
            loss_val = criterion(outputs, labels)
            test_loss += loss_val.item()
    test_loss /= len(test_loader)

    print(f'Epocha {epocha+1}, Trénovací Loss: {train_loss:.4f}, Testovací Loss: {test_loss:.4f}')

    # ------------------------------
    # Kontrola pro Early Stopping
    # ------------------------------
    if best_loss - test_loss > threshold:
        best_loss = test_loss
        epochs_no_improve = 0
    else:
        epochs_no_improve += 1

    if epochs_no_improve >= patience:
        print(f"Early stopping aktivován po {epocha+1} epochách.")
        break

Epocha 1, Trénovací Loss: 1.9534, Testovací Loss: 1.6244
Epocha 2, Trénovací Loss: 1.5255, Testovací Loss: 1.3394
Epocha 3, Trénovací Loss: 1.3770, Testovací Loss: 1.3196
Epocha 4, Trénovací Loss: 1.2624, Testovací Loss: 1.2182
Epocha 5, Trénovací Loss: 1.2094, Testovací Loss: 1.1687
Epocha 6, Trénovací Loss: 1.1184, Testovací Loss: 1.1701
Epocha 7, Trénovací Loss: 1.0546, Testovací Loss: 1.1111
Epocha 8, Trénovací Loss: 0.9850, Testovací Loss: 1.1399
Epocha 9, Trénovací Loss: 0.9293, Testovací Loss: 1.0759
Epocha 10, Trénovací Loss: 0.8339, Testovací Loss: 1.2486
Epocha 11, Trénovací Loss: 0.8166, Testovací Loss: 1.2048
Epocha 12, Trénovací Loss: 0.6988, Testovací Loss: 1.2834
Early stopping aktivován po 12 epochách.


**6. Evaluation on the test set**

In [10]:
correct = 0
total = 0
with torch.no_grad():  # Disable gradient calculation for evaluation
    for images, labels in test_loader:
        # Move images and labels to the device
        images = images.to(device)
        labels = labels.to(device)

        outputs = model(images)
        _, predicted = torch.max(outputs, 1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()

print(f'Accuracy: {100 * correct / total:.2f}%')

Accuracy: 53.11%
